# 📊 US DOT Flight Operations — Data Profile

> **Phase 3 Demo Dataset** | Fabric Lakehouse: `airline_flights`
> 
> Kaggle US DOT Flight Delays (2015, CC0 Public Domain)
> - `airlines`: 14 airline codes
> - `airports`: 322 airport records  
> - `flights`: 5,819,079 flight records
>
> **Purpose:** Data source profiling for the Fabric (structured) + Foundry IQ (unstructured) Semantic JOIN demo

In [ ]:
# Check available tables
display(spark.sql("SHOW TABLES"))

## 2. Flights Data Preview
> Complete 2015 US domestic flight records. Each row represents a single flight,
> including departure/arrival delays (minutes), cancellation status, and cancellation reason.
> 
> - **DEPARTURE_DELAY**: Positive = delayed (minutes), Negative = departed early
> - **CANCELLED**: 0 = operated, 1 = cancelled
> - **CANCELLATION_REASON**: A=Airline, B=Weather, C=NAS, D=Security

In [ ]:
display(spark.sql("SELECT * FROM flights LIMIT 5"))

## 3. Airline Delay & Cancellation Statistics

> Comparing total flights, average departure delay (minutes), and cancellation count across 14 airlines.

> These statistics are the "numbers" part that Fabric provides in the Phase 3 Semantic JOIN.

In [ ]:
display(spark.sql("""
    SELECT a.AIRLINE as airline_name,
           f.AIRLINE as code,
           COUNT(*) as total_flights,
           ROUND(AVG(f.DEPARTURE_DELAY),1) as avg_delay_min,
           SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled_count,
           ROUND(SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as cancel_rate_pct
    FROM flights f
    JOIN airlines a ON f.AIRLINE = a.IATA_CODE
    GROUP BY a.AIRLINE, f.AIRLINE
    ORDER BY avg_delay_min DESC
"""))

## 4. Demo Scenario Validation: Semantic JOIN

> **Example Question:** "How many JFK flights were delayed over 2 hours, and what is the compensation policy?"

> This question requires **two types of data sources** to produce a single answer.
 
| Part | Source | Role |
|------|--------|------|
| Numbers | Fabric Lakehouse (flights table) | "How many?" → SQL aggregation |
| Explanation | SharePoint PDF (DOT aviation regulations) | "What is the policy?" → Document search |
 
> The query below verifies the actual result for the **numbers part**.

> The combination of these numbers and document search results into a single answer via LLM is the **Semantic JOIN**.

In [ ]:
display(spark.sql("""
    SELECT COUNT(*) as jfk_delayed_over_2hrs
    FROM flights
    WHERE ORIGIN_AIRPORT = 'JFK' AND DEPARTURE_DELAY >= 120
"""))

## 5. Fabric IQ — Aggregated Stats for AI Search Indexing

5.8M raw rows cannot be directly indexed by the AI Search OneLake Indexer (CSV/Delta Parquet not supported).
Instead, we aggregate the data using Spark SQL and convert the results to JSON documents stored in OneLake Files.

These JSON files are indexed by AI Search and become Foundry IQ's Knowledge Source (KS-1).

- `airline_delay_stats.json` — Delay/cancellation stats for 14 airlines
- `top_airport_stats.json` — Delay/cancellation stats for top 30 airports
- `monthly_trend.json` — Monthly flight trends
- `cancellation_reasons.json` — Cancellation reason distribution
- `jfk_detailed_analysis.json` — JFK airport detailed analysis (for demo questions)

Output: `Files/stats/*.json` → OneLake Indexer → AI Search → unified-airline-kb KS-1

In [ ]:
import json

# Airline delay statistics
airline_stats = spark.sql("""
    SELECT a.AIRLINE as airline_name, f.AIRLINE as code,
           COUNT(*) as total_flights,
           ROUND(AVG(f.DEPARTURE_DELAY),1) as avg_delay_min,
           SUM(CASE WHEN f.DEPARTURE_DELAY >= 120 THEN 1 ELSE 0 END) as delayed_2hrs_plus,
           SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled,
           ROUND(SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) as cancel_rate_pct
    FROM flights f JOIN airlines a ON f.AIRLINE = a.IATA_CODE
    GROUP BY a.AIRLINE, f.AIRLINE ORDER BY avg_delay_min DESC
""").toPandas()

# Airport delay statistics (top 30)
airport_stats = spark.sql("""
    SELECT ap.AIRPORT as airport_name, f.ORIGIN_AIRPORT as code,
           ap.CITY, ap.STATE,
           COUNT(*) as total_departures,
           ROUND(AVG(f.DEPARTURE_DELAY),1) as avg_delay_min,
           SUM(CASE WHEN f.DEPARTURE_DELAY >= 120 THEN 1 ELSE 0 END) as delayed_2hrs_plus,
           SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled
    FROM flights f JOIN airports ap ON f.ORIGIN_AIRPORT = ap.IATA_CODE
    GROUP BY ap.AIRPORT, f.ORIGIN_AIRPORT, ap.CITY, ap.STATE
    ORDER BY total_departures DESC LIMIT 30
""").toPandas()

# Monthly trend
monthly_stats = spark.sql("""
    SELECT MONTH, COUNT(*) as total_flights,
           ROUND(AVG(DEPARTURE_DELAY),1) as avg_delay_min,
           SUM(CASE WHEN CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled,
           SUM(CASE WHEN DEPARTURE_DELAY >= 120 THEN 1 ELSE 0 END) as delayed_2hrs_plus
    FROM flights GROUP BY MONTH ORDER BY MONTH
""").toPandas()

# Cancellation reason statistics
cancel_stats = spark.sql("""
    SELECT CANCELLATION_REASON as reason,
           COUNT(*) as count,
           CASE CANCELLATION_REASON
             WHEN 'A' THEN 'Airline'
             WHEN 'B' THEN 'Weather'
             WHEN 'C' THEN 'National Air System'
             WHEN 'D' THEN 'Security'
           END as reason_label
    FROM flights WHERE CANCELLED = 1
    GROUP BY CANCELLATION_REASON ORDER BY count DESC
""").toPandas()

# JFK detailed analysis
jfk_stats = spark.sql("""
    SELECT a.AIRLINE as airline_name, f.AIRLINE as code,
           COUNT(*) as total_flights,
           ROUND(AVG(f.DEPARTURE_DELAY),1) as avg_delay,
           SUM(CASE WHEN f.DEPARTURE_DELAY >= 120 THEN 1 ELSE 0 END) as delayed_2hrs_plus,
           SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled
    FROM flights f JOIN airlines a ON f.AIRLINE = a.IATA_CODE
    WHERE f.ORIGIN_AIRPORT = 'JFK'
    GROUP BY a.AIRLINE, f.AIRLINE ORDER BY total_flights DESC
""").toPandas()

# Save as JSON files
stats = {
    "airline_delay_stats": json.loads(airline_stats.to_json(orient='records')),
    "top_airport_stats": json.loads(airport_stats.to_json(orient='records')),
    "monthly_trend": json.loads(monthly_stats.to_json(orient='records')),
    "cancellation_reasons": json.loads(cancel_stats.to_json(orient='records')),
    "jfk_detailed_analysis": json.loads(jfk_stats.to_json(orient='records'))
}

# Save to OneLake Files/stats/ folder
import os
os.makedirs('/lakehouse/default/Files/stats', exist_ok=True)

for name, data in stats.items():
    doc = {
        "title": name.replace('_', ' ').title(),
        "source": "Fabric Lakehouse - airline_flights (5.8M US DOT flights 2015)",
        "data": data,
        "summary": f"Aggregated {name} from 5,819,079 US domestic flight records (2015)"
    }
    with open(f'/lakehouse/default/Files/stats/{name}.json', 'w') as f:
        json.dump(doc, f, indent=2)
    print(f"✅ {name}.json saved ({len(data)} records)")

print(f"\n✅ All stats exported to Files/stats/")

## 6. EDA — Visualizations from Aggregated Stats

Visualize key patterns based on the aggregated JSON data generated above.
Each chart is saved as an individual PNG file for use in the HTML export.

- 📊 Chart 1: Average delay distribution by airline ← `airline_delay_stats.json`
- 📊 Chart 2: Delay pattern by hour of day ← hourly aggregation from `flights`
- 📊 Chart 3: Cancellation reason distribution ← `cancellation_reasons.json`
- 📊 Chart 4: Cancellation rate by airline ← `airline_delay_stats.json`

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

# Spark → Pandas (aggregated — memory safe)
df_airline_stats = spark.sql("""
    SELECT a.AIRLINE as airline_name, f.AIRLINE as code,
           COUNT(*) as total_flights,
           ROUND(AVG(f.DEPARTURE_DELAY),1) as avg_delay,
           ROUND(AVG(CASE WHEN f.DEPARTURE_DELAY > 0 THEN f.DEPARTURE_DELAY END),1) as avg_delay_when_late,
           SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END) as cancelled,
           ROUND(SUM(CASE WHEN f.CANCELLED = 1 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) as cancel_rate
    FROM flights f JOIN airlines a ON f.AIRLINE = a.IATA_CODE
    GROUP BY a.AIRLINE, f.AIRLINE
    ORDER BY avg_delay DESC
""").toPandas()

df_hourly = spark.sql("""
    SELECT CAST(DEPARTURE_TIME / 100 AS INT) as hour,
           COUNT(*) as flights,
           ROUND(AVG(DEPARTURE_DELAY),1) as avg_delay
    FROM flights
    WHERE DEPARTURE_TIME IS NOT NULL AND DEPARTURE_TIME > 0
    GROUP BY CAST(DEPARTURE_TIME / 100 AS INT)
    ORDER BY hour
""").toPandas()

df_cancel_reason = spark.sql("""
    SELECT CANCELLATION_REASON as reason, COUNT(*) as count
    FROM flights WHERE CANCELLED = 1
    GROUP BY CANCELLATION_REASON
""").toPandas()
df_cancel_reason['label'] = df_cancel_reason['reason'].map({
    'A': 'Airline', 'B': 'Weather', 'C': 'NAS', 'D': 'Security'
})

df_airline_stats['short_name'] = df_airline_stats['airline_name'].str.split(' Air').str[0]

import os
os.makedirs('/lakehouse/default/Files/stats', exist_ok=True)
print("✅ Data prepared for visualization")

### 📊 Chart 1: Average Delay by Airline
> Source: `airline_delay_stats.json` — Average departure delay ranking across 14 airlines.
> Spirit Airlines (NK) has the highest delay, Hawaiian Airlines (HA) has the lowest.

In [ ]:
# 📊 Chart 1: Average Delay by Airline
# Source: airline_delay_stats.json (14 airlines)

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette("RdYlGn_r", len(df_airline_stats))
ax.barh(df_airline_stats['short_name'], df_airline_stats['avg_delay'], color=colors)
ax.set_xlabel('Avg Departure Delay (min)')
ax.set_title('Average Delay by Airline\nSource: airline_delay_stats.json', fontsize=14, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()

plt.savefig('/lakehouse/default/Files/stats/chart_avg_delay_by_airline.png', dpi=150, bbox_inches='tight')
display(fig)
print("✅ chart_avg_delay_by_airline.png saved")

### 📊 Chart 2: Delay Pattern by Hour
> Source: hourly aggregation from `flights` table — Average delay pattern by time of day.
> Delays spike during the 3–6 PM rush hour; 5 AM has the highest on-time departure rate.

In [ ]:
# 📊 Chart 2: Delay Pattern by Hour
# Source: hourly aggregation from flights table

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(df_hourly['hour'], df_hourly['avg_delay'], alpha=0.3, color='#e74c3c')
ax.plot(df_hourly['hour'], df_hourly['avg_delay'], color='#e74c3c', linewidth=2)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Avg Delay (min)')
ax.set_title('Delay Pattern by Hour\nSource: hourly aggregation from flights table', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.savefig('/lakehouse/default/Files/stats/chart_delay_by_hour.png', dpi=150, bbox_inches='tight')
display(fig)
print("✅ chart_delay_by_hour.png saved")

### 📊 Chart 3: Cancellation Reasons
> Source: `cancellation_reasons.json` — Cancellation reason distribution.
> Weather accounts for 54.3% (highest), followed by Airline (28.1%) and NAS (17.5%).

In [ ]:
# 📊 Chart 3: Cancellation Reasons
# Source: cancellation_reasons.json (4 reasons)

fig, ax = plt.subplots(figsize=(8, 8))
colors_pie = ['#e74c3c', '#3498db', '#f39c12', '#2ecc71']
ax.pie(df_cancel_reason['count'], labels=df_cancel_reason['label'],
       autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax.set_title('Cancellation Reasons\nSource: cancellation_reasons.json', fontsize=14, fontweight='bold')
plt.tight_layout()

plt.savefig('/lakehouse/default/Files/stats/chart_cancellation_reasons.png', dpi=150, bbox_inches='tight')
display(fig)
print("✅ chart_cancellation_reasons.png saved")

### 📊 Chart 4: Cancellation Rate by Airline
> Source: `airline_delay_stats.json` — Cancellation rate ranking by airline.
> American Eagle (MQ) has the highest cancellation rate, United (UA) has the lowest.

In [ ]:
# 📊 Chart 4: Cancellation Rate by Airline
# Source: airline_delay_stats.json (14 airlines)

fig, ax = plt.subplots(figsize=(10, 7))
df_sorted = df_airline_stats.sort_values('cancel_rate')
df_sorted['short_name'] = df_sorted['airline_name'].str.split(' Air').str[0]
colors2 = sns.color_palette("RdYlGn_r", len(df_sorted))
ax.barh(df_sorted['short_name'], df_sorted['cancel_rate'], color=colors2)
ax.set_xlabel('Cancellation Rate (%)')
ax.set_title('Cancellation Rate by Airline\nSource: airline_delay_stats.json', fontsize=14, fontweight='bold')
plt.tight_layout()

plt.savefig('/lakehouse/default/Files/stats/chart_cancellation_rate.png', dpi=150, bbox_inches='tight')
display(fig)
print("✅ chart_cancellation_rate.png saved")

## 7. HTML Export

Export the data profile charts to HTML for use as the data introduction page in the Knowledge Retrieval Studio app.
The 4 individual chart images are arranged in a 2×2 grid layout.

In [ ]:
import base64

# Convert 4 individual chart images to base64
chart_files = [
    ('Average Delay by Airline', 'airline_delay_stats.json', 'chart_avg_delay_by_airline.png'),
    ('Delay Pattern by Hour', 'monthly_trend.json', 'chart_delay_by_hour.png'),
    ('Cancellation Reasons', 'cancellation_reasons.json', 'chart_cancellation_reasons.png'),
    ('Cancellation Rate by Airline', 'airline_delay_stats.json', 'chart_cancellation_rate.png'),
]

chart_html = ""
for title, source, filename in chart_files:
    filepath = f'/lakehouse/default/Files/stats/{filename}'
    with open(filepath, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode()
    chart_html += f'''
    <div style="margin-bottom: 2rem;">
      <h4 style="margin-bottom: 4px;">📊 {title}</h4>
      <p style="font-size: 12px; color: #888; font-family: 'JetBrains Mono', monospace; margin-bottom: 8px;">
        Source: {source}
      </p>
      <img src="data:image/png;base64,{img_b64}" style="width:100%; border-radius: 8px;" />
    </div>
    '''

# Stats
total_flights = df_airline_stats['total_flights'].sum()
avg_delay_all = round(df_airline_stats['avg_delay'].mean(), 1)
total_cancelled = df_airline_stats['cancelled'].sum()

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Flight Data Profile — Knowledge Retrieval Studio</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;700&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg: #08080c; --surface: rgba(255,255,255,0.03); --surface-border: rgba(255,255,255,0.06);
    --text: #e0e0e0; --text-dim: #888; --text-muted: #555;
    --accent: #60a5fa; --accent-glow: rgba(96,165,250,0.15);
    --green: #34d399; --purple: #a78bfa; --orange: #fb923c;
  }}
  * {{ margin: 0; padding: 0; box-sizing: border-box; }}
  body {{ background: var(--bg); color: var(--text); font-family: 'DM Sans', sans-serif; line-height: 1.6; -webkit-font-smoothing: antialiased; }}
  .container {{ max-width: 1100px; margin: 0 auto; padding: 48px 24px 64px; }}
  .badge {{ display: inline-flex; align-items: center; gap: 6px; background: var(--accent-glow); border: 1px solid rgba(96,165,250,0.2); color: var(--accent); font-size: 0.75rem; font-weight: 500; padding: 4px 12px; border-radius: 100px; margin-bottom: 16px; font-family: 'JetBrains Mono', monospace; }}
  .badge::before {{ content: ''; width: 6px; height: 6px; background: var(--accent); border-radius: 50%; }}
  h1 {{ font-size: 2.25rem; font-weight: 700; letter-spacing: -0.02em; margin-bottom: 8px; }}
  h1 span {{ color: var(--accent); }}
  .subtitle {{ color: var(--text-dim); font-size: 1.05rem; }}
  .subtitle code {{ font-family: 'JetBrains Mono', monospace; font-size: 0.9rem; background: rgba(255,255,255,0.06); padding: 2px 8px; border-radius: 4px; color: var(--accent); }}
  .stats {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin: 32px 0; }}
  .stat-card {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 12px; padding: 20px; text-align: center; }}
  .stat-card .number {{ font-size: 1.75rem; font-weight: 700; font-family: 'JetBrains Mono', monospace; color: var(--accent); }}
  .stat-card .label {{ color: var(--text-dim); font-size: 0.85rem; margin-top: 4px; }}
  section {{ margin-bottom: 40px; }}
  h2 {{ font-size: 1.25rem; font-weight: 600; margin-bottom: 8px; }}
  .section-desc {{ color: var(--text-dim); font-size: 0.95rem; margin-bottom: 20px; max-width: 720px; }}
  .context-box {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 12px; padding: 24px; margin-bottom: 24px; }}
  .context-box p {{ color: var(--text-dim); font-size: 0.9rem; margin-bottom: 10px; line-height: 1.7; }}
  .highlight-quote {{ background: linear-gradient(135deg, rgba(96,165,250,0.08), rgba(167,139,250,0.08)); border-left: 3px solid var(--accent); padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 16px 0; font-size: 1rem; font-weight: 500; }}
  .join-diagram {{ display: grid; grid-template-columns: 1fr auto 1fr; gap: 16px; align-items: center; margin: 24px 0; }}
  .join-source {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 12px; padding: 20px; }}
  .join-source .source-label {{ font-family: 'JetBrains Mono', monospace; font-size: 0.7rem; text-transform: uppercase; letter-spacing: 0.1em; margin-bottom: 8px; }}
  .join-source.fabric .source-label {{ color: var(--green); }}
  .join-source.foundry .source-label {{ color: var(--purple); }}
  .join-source .source-title {{ font-weight: 600; font-size: 0.95rem; margin-bottom: 4px; }}
  .join-source .source-detail {{ color: var(--text-dim); font-size: 0.85rem; }}
  .join-source .source-result {{ margin-top: 12px; padding-top: 12px; border-top: 1px solid var(--surface-border); font-family: 'JetBrains Mono', monospace; font-size: 0.85rem; }}
  .join-source.fabric .source-result {{ color: var(--green); }}
  .join-source.foundry .source-result {{ color: var(--purple); }}
  .join-arrow {{ display: flex; flex-direction: column; align-items: center; gap: 4px; color: var(--text-muted); font-size: 0.75rem; font-family: 'JetBrains Mono', monospace; }}
  .join-arrow .arrow-icon {{ font-size: 1.5rem; color: var(--orange); }}
  .demo-query {{ background: linear-gradient(135deg, rgba(96,165,250,0.06), rgba(167,139,250,0.06)); border: 1px solid rgba(96,165,250,0.15); border-radius: 12px; padding: 24px; margin-top: 24px; }}
  .demo-query .query-text {{ font-size: 1.05rem; font-weight: 500; margin-bottom: 16px; }}
  .demo-query .query-text::before {{ content: 'Q. '; color: var(--accent); font-weight: 700; }}
  .demo-query .answer-preview {{ background: rgba(0,0,0,0.3); border-radius: 8px; padding: 16px; font-size: 0.9rem; line-height: 1.7; color: var(--text-dim); }}
  .cite {{ display: inline-flex; align-items: center; justify-content: center; width: 18px; height: 18px; border-radius: 4px; font-size: 0.65rem; font-weight: 700; margin: 0 2px; vertical-align: middle; font-family: 'JetBrains Mono', monospace; }}
  .cite-fabric {{ background: rgba(52,211,153,0.2); color: var(--green); }}
  .cite-foundry {{ background: rgba(167,139,250,0.2); color: var(--purple); }}
  .chart-grid {{ display: grid; grid-template-columns: 1fr 1fr; gap: 1.5rem; }}
  .chart-wrap {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 12px; padding: 16px; }}
  .chart-wrap img {{ width: 100%; border-radius: 8px; }}
  .data-table-wrap {{ background: var(--surface); border: 1px solid var(--surface-border); border-radius: 12px; overflow: hidden; }}
  .data-table {{ width: 100%; border-collapse: collapse; font-size: 0.85rem; }}
  .data-table thead {{ background: rgba(255,255,255,0.04); }}
  .data-table th {{ text-align: left; padding: 12px 16px; font-weight: 600; font-size: 0.75rem; text-transform: uppercase; letter-spacing: 0.05em; color: var(--text-dim); border-bottom: 1px solid var(--surface-border); }}
  .data-table td {{ padding: 10px 16px; border-bottom: 1px solid rgba(255,255,255,0.03); font-family: 'JetBrains Mono', monospace; font-size: 0.8rem; }}
  .tag {{ display: inline-block; padding: 2px 8px; border-radius: 4px; font-size: 0.7rem; font-weight: 500; font-family: 'JetBrains Mono', monospace; }}
  .tag-int {{ background: rgba(96,165,250,0.15); color: var(--accent); }}
  .tag-str {{ background: rgba(52,211,153,0.15); color: var(--green); }}
  .footer {{ text-align: center; color: var(--text-muted); font-size: 0.8rem; margin-top: 48px; padding-top: 24px; border-top: 1px solid var(--surface-border); }}
  @media (max-width: 600px) {{ .stats {{ grid-template-columns: repeat(2, 1fr); }} .join-diagram {{ grid-template-columns: 1fr; }} .chart-grid {{ grid-template-columns: 1fr; }} }}
</style>
</head>
<body>
<div class="container">
  <div class="badge">PHASE 3 · FABRIC + FOUNDRY IQ</div>
  <h1>📊 <span>Flight Data</span> Profile</h1>
  <p class="subtitle">US DOT Flight Operations 2015 — Fabric Lakehouse: <code>airline_flights</code></p>

  <div class="stats">
    <div class="stat-card"><div class="number">{total_flights:,}</div><div class="label">Total Flights</div></div>
    <div class="stat-card"><div class="number">14</div><div class="label">Airlines</div></div>
    <div class="stat-card"><div class="number">{avg_delay_all}</div><div class="label">Avg Delay (min)</div></div>
    <div class="stat-card"><div class="number">{total_cancelled:,}</div><div class="label">Cancelled</div></div>
  </div>

  <section>
    <h2>What is this data?</h2>
    <div class="context-box">
      <p>This dataset contains <strong>every US domestic flight in 2015</strong> — 5.8 million records from 14 airlines across 322 airports. Each row represents a single flight with departure/arrival delays, cancellation status, and routing information.</p>
      <p>The data is sourced from the <strong>US Department of Transportation Bureau of Transportation Statistics</strong> and hosted on Kaggle under CC0 Public Domain license.</p>
      <p>It is loaded into a <strong>Microsoft Fabric Lakehouse</strong> as Delta Tables, enabling Spark SQL queries and OneLake indexing for AI Search integration.</p>
    </div>
  </section>

  <section>
    <h2>One Question, Two Brains</h2>
    <div class="context-box">
      <div class="highlight-quote">"Numbers come from Fabric, reasons come from Foundry IQ."</div>
      <p>Phase 3 demonstrates <strong>Semantic JOIN</strong> — a single question answered by combining structured data (this flight dataset from Fabric) with unstructured documents (DOT regulation PDFs indexed from SharePoint via Foundry IQ).</p>
      <p>This is not a SQL JOIN. The LLM reasons across both sources and synthesizes a unified answer with citations from each.</p>
    </div>

    <div class="join-diagram">
      <div class="join-source fabric">
        <div class="source-label">Fabric IQ — Structured</div>
        <div class="source-title">flights table (5.8M rows)</div>
        <div class="source-detail">Delay counts, cancellation stats, route data</div>
        <div class="source-result">→ "2,566 flights delayed 2+ hrs at JFK"</div>
      </div>
      <div class="join-arrow">
        <div class="arrow-icon">⚡</div>
        <div>Semantic<br>JOIN</div>
      </div>
      <div class="join-source foundry">
        <div class="source-label">Foundry IQ — Unstructured</div>
        <div class="source-title">DOT Policy PDFs (4 docs)</div>
        <div class="source-detail">Passenger rights, compensation rules, regulations</div>
        <div class="source-result">→ "Full refund for 3+ hr delays per DOT"</div>
      </div>
    </div>

    <div class="demo-query">
      <div class="query-text">How many JFK flights were delayed over 2 hours, and what's the compensation policy?</div>
      <div class="answer-preview">
        JFK departures delayed over 2 hours totaled <strong>2,566 flights</strong> in 2015 <span class="cite cite-fabric">1</span>. 
        Under DOT regulations, passengers on domestic flights delayed 3 or more hours are entitled to a <strong>full refund</strong> if they choose not to travel <span class="cite cite-foundry">2</span>.
        For delays between 1-2 hours involving involuntary denied boarding, compensation is 200% of the one-way fare <span class="cite cite-foundry">2</span>.
      </div>
    </div>
  </section>

  <section>
    <h2>Key Schema</h2>
    <div class="data-table-wrap">
      <table class="data-table">
        <thead><tr><th>Column</th><th>Type</th><th>Example</th><th>Description</th></tr></thead>
        <tbody>
          <tr><td>AIRLINE</td><td><span class="tag tag-str">str</span></td><td>AA</td><td>IATA airline code</td></tr>
          <tr><td>ORIGIN_AIRPORT</td><td><span class="tag tag-str">str</span></td><td>JFK</td><td>Departure airport</td></tr>
          <tr><td>DESTINATION_AIRPORT</td><td><span class="tag tag-str">str</span></td><td>LAX</td><td>Arrival airport</td></tr>
          <tr><td>DEPARTURE_DELAY</td><td><span class="tag tag-int">int</span></td><td>45</td><td>Minutes delayed (negative = early)</td></tr>
          <tr><td>ARRIVAL_DELAY</td><td><span class="tag tag-int">int</span></td><td>42</td><td>Minutes delayed at arrival</td></tr>
          <tr><td>CANCELLED</td><td><span class="tag tag-int">int</span></td><td>0 / 1</td><td>0 = operated, 1 = cancelled</td></tr>
          <tr><td>CANCELLATION_REASON</td><td><span class="tag tag-str">str</span></td><td>B</td><td>A=Airline, B=Weather, C=NAS, D=Security</td></tr>
          <tr><td>DISTANCE</td><td><span class="tag tag-int">int</span></td><td>2475</td><td>Flight distance in miles</td></tr>
        </tbody>
      </table>
    </div>
  </section>

  <section>
    <h2>EDA Visualizations</h2>
    <p class="section-desc">Generated by Fabric Notebook (PySpark + Seaborn) from the airline_flights Lakehouse. Each chart maps to an aggregated JSON file in <code>Files/stats/</code>.</p>
    <div class="chart-grid">
      {chart_html}
    </div>
  </section>

  <p class="footer">
    Source: <a href="https://www.kaggle.com/datasets/usdot/flight-delays">Kaggle US DOT Flight Delays</a> (CC0) · 
    Generated by Fabric Notebook · 
    <a href="https://foundry-iq-demo-suite.vercel.app">Knowledge Retrieval Studio</a>
  </p>
</div>
</body>
</html>"""

with open('/lakehouse/default/Files/flight_data_profile.html', 'w') as f:
    f.write(html)

print(f"✅ HTML exported to Files/flight_data_profile.html")
print(f"   Total flights: {total_flights:,}")
print(f"   Avg delay: {avg_delay_all} min")
print(f"   Cancelled: {total_cancelled:,}")
print(f"   Charts: {len(chart_files)} individual images in 2×2 grid")